Här är en sammanfattning som också kan fungera som en **installationsguide för en annan gruppmedlem**. 

# Utvecklingsmiljö för projektet – Windows + WSL + Docker

## Målet

Vi satte upp en utvecklingsmiljö där projektet kan utvecklas från **Ubuntu i WSL 2**, samtidigt som Docker Desktop kör och hanterar projektets containers.

Den färdiga strukturen ser förenklat ut så här:

```text
Windows
│
├── Docker Desktop
│       │
│       └── Docker Engine
│
└── WSL 2
     │
     └── Ubuntu
          │
          ├── Git
          ├── Python
          ├── Poetry
          ├── Make
          ├── curl
          ├── Docker CLI
          └── VS Code
                │
                └── Projektet
```

Airflow, PostgreSQL, MinIO m.m. ska senare i första hand **köras i Docker-containers**, inte installeras direkt i Ubuntu.

---



# 1. Kontrollera att Ubuntu använder WSL 2

I **PowerShell**:

```powershell
wsl -l -v
```

### Varför?

WSL låter oss använda Linux/Ubuntu på en Windows-dator. Projektet kommer använda Linuxverktyg och Docker, och därför vill vi använda **WSL 2**.

### Förväntat resultat

Något liknande:

```text
NAME       STATE      VERSION
Ubuntu     Running    2
```

Det viktiga är:

```text
VERSION
2
```

---



# 2. Uppdatera WSL

I **PowerShell**:

```powershell
wsl --update
```

Kontrollera vid behov:

```powershell
wsl --version
```

### Varför?

Docker Desktop integrerar med WSL. En aktuell WSL-version minskar risken för kompatibilitetsproblem.

### Resultat

WSL ska rapportera att den är uppdaterad eller installera en nyare version.

---



# 3. Installera Docker Desktop

Installera **Docker Desktop for Windows**.

Docker Desktop installeras alltså i **Windows**, inte med `apt` inne i Ubuntu.

### Varför?

Docker ska senare köra projektets olika tjänster isolerat i containers, exempelvis:

```text
Docker
├── Airflow
├── PostgreSQL
├── MinIO
└── eventuellt Ollama
```

Det gör att gruppmedlemmarna kan använda samma programversioner och miljöer trots att deras egna datorer skiljer sig åt.

---



# 4. Ställ in Docker på WSL 2

I Docker Desktop:

**Settings → General**

Kontrollera:

```text
Linux backend type
WSL 2
```

### Varför?

Det gör att Docker använder WSL 2 som Linuxmiljö istället för en separat lösning.

---



# 5. Aktivera Docker-integrationen med Ubuntu

I Docker Desktop:

**Settings → Resources → WSL Integration**

Aktivera:

```text
Enable integration with my default WSL distro
```

Om Ubuntu inte är standarddistribution kan den istället väljas bland ytterligare distributioner.

### Varför?

Docker Desktop ligger på Windows-sidan, medan vi vill arbeta från Ubuntu.

Integrationen gör alltså ungefär:

```text
Ubuntu / WSL
      ↓
docker-kommandon
      ↓
Docker Desktop
      ↓
Docker Engine
```

---



# 6. Kontrollera Docker från Ubuntu

Nu går vi över till **Ubuntu-terminalen**.

Kör:

```bash
docker --version
```

### Förväntat resultat

```text
Docker version ...
```

Det visar att Ubuntu hittar Docker.

---



# 7. Kontrollera Docker Compose

I Ubuntu:

```bash
docker compose version
```

### Förväntat resultat

```text
Docker Compose version ...
```

### Varför?

Projektets mål är att hela systemet så småningom ska kunna startas med:

```bash
docker compose up
```

Compose används för att definiera och starta **flera containers som hör ihop**.

Viktigt: det moderna kommandot är:

```bash
docker compose
```

inte det äldre:

```bash
docker-compose
```

---



# 8. Testa att Docker verkligen kan köra en container

Kör:

```bash
docker run hello-world
```

Första gången hämtar Docker imagen från Docker Hub.

### Förväntat resultat

Bland annat:

```text
Hello from Docker!

This message shows that your installation appears to be working correctly.
```

### Varför?

`docker --version` visar egentligen bara att kommandot finns.

`hello-world` testar hela kedjan:

```text
Docker CLI
   ↓
Docker Engine
   ↓
hämta image
   ↓
skapa container
   ↓
starta container
   ↓
returnera output
```

Så detta är en viktigare kontroll.

---



# 9. Uppdatera Ubuntus paketlista

Kör:

```bash
sudo apt update
```

Ubuntu kan fråga efter Linux-användarens lösenord.

### Varför?

`apt` är Ubuntus pakethanterare. Kommandot uppdaterar informationen om vilka paket och versioner som finns tillgängliga.

Det kan efteråt stå exempelvis:

```text
27 packages can be upgraded.
```

Det är bara information. Det betyder inte att något gått fel.

---



# 10. Installera grundläggande utvecklingsverktyg

Kör:

```bash
sudo apt install -y build-essential curl
```

### Varför?

**`build-essential`** installerar vanliga utvecklings-/byggverktyg som kan behövas av projekt och Pythonpaket.

**`curl`** används för att hämta data/filer via terminalen.

Installationen ger också tillgång till **Make**.

---



# 11. Kontrollera Make

Kör:

```bash
make --version
```

### Förväntat resultat

Exempel:

```text
GNU Make 4.x.x
```

### Varför?

Projektet innehåller en:

```text
Makefile
```

Den kan innehålla genvägar till längre utvecklingskommandon.

Istället för flera kommandon kan projektet exempelvis låta utvecklaren köra något i stil med:

```bash
make install_dependencies
```

Make är dock ett hjälpredskap – det är inte samma sak som Docker eller Airflow.

---



# 12. Kontrollera Python

Kör:

```bash
python3 --version
```

### Varför?

Python används av projektets kod och av Poetry.

I vår miljö fanns en **nyare Pythonversion** än legacyprojektet kräver.

Projektets `pyproject.toml` innehåller:

```toml
python = "~3.10"
```

Det betyder att **den befintliga legacyversionen av projektet förväntar sig Python 3.10.x**.

### Viktigt

Vi installerade **inte Python 3.10**.

Anledningen är att projektuppgiften går ut på att modernisera bland annat Python/Airflow. Gruppen bör därför först bestämma vilken kombination som den nya lösningen ska använda.

Docker kan dessutom låta projektet använda en annan Pythonversion än den som finns installerad direkt i Ubuntu.

---



# 13. Installera Poetry

Kör:

```bash
curl -sSL https://install.python-poetry.org | python3 -
```

### Varför?

Projektet använder Poetry för att hantera Pythonprojektets dependencies och versioner.

Projektet innehåller exempelvis:

```text
pyproject.toml
poetry.lock
```

Förenklat:

```text
pyproject.toml
→ beskriver projekt och dependencies

poetry.lock
→ låser exakta dependency-versioner

Poetry
→ installerar/hanterar dem
```

---



# 14. Lägg Poetry i PATH

Poetry installerades i användarens lokala bin-katalog, vanligtvis:

```text
~/.local/bin
```

För att kunna skriva `poetry` direkt i terminalen lades katalogen till i `PATH`:

```bash
echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc
```

Ladda sedan om Bash-konfigurationen:

```bash
source ~/.bashrc
```

### Vad är PATH?

`PATH` är listan över kataloger där Linux letar efter program när man skriver ett kommando.

När man skriver:

```bash
poetry
```

kan Ubuntu tack vare `PATH` hitta själva Poetry-programmet.

---



# 15. Kontrollera Poetry

Kör:

```bash
poetry --version
```

### Förväntat resultat

```text
Poetry (version ...)
```

Då fungerar Poetry från Ubuntu.

---



# 16. Kontrollera VS Code från WSL

Kör:

```bash
code --version
```

### Förväntat resultat

VS Codes versionsnummer.

### Varför?

Vi vill kunna öppna projektet **från Ubuntu**, inte arbeta med Linuxprojektet som om det vore ett vanligt Windowsprojekt.

Gå till projektmappen:

```bash
cd /sökväg/till/projektet
```

Öppna sedan:

```bash
code .
```

Punkten betyder ungefär:

> Öppna den katalog jag står i just nu.

---



# 17. Kontrollera att VS Code verkligen använder WSL

När projektet öppnas ska VS Code visa något i stil med:

```text
[WSL: Ubuntu]
```

Det kan exempelvis synas längst ner till vänster och i fönstrets titel.

### Varför?

Det visar att:

```text
VS Code
   ↓
Ubuntu/WSL
   ↓
projektfiler
```

används som utvecklingsmiljö.

Terminalen inne i VS Code ska därmed också vara en Linuxterminal.

---



# 18. Kontrollera Docker från projektets VS Code-terminal

Kör:

```bash
docker ps
```

### Förväntat resultat

En tabell ungefär:

```text
CONTAINER ID   IMAGE   COMMAND   CREATED   STATUS   PORTS   NAMES
```

Det gör inget om tabellen är tom.

### Varför?

Det bekräftar att Docker fungerar även från **den faktiska utvecklingsmiljön i VS Code/WSL**.

En tom lista betyder bara:

> Docker fungerar, men inga containers körs just nu.

---



# 19. Kontrollera Git-repot

Från projektmappen:

```bash
git status
```

### Ett bra resultat kan exempelvis vara:

```text
On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean
```

### Vad betyder det?

`On branch master`
→ vi står på Git-branchen `master`.

`up to date with origin/master`
→ vår lokala version är synkad med remote-repot.

`working tree clean`
→ vi har inga osparade Git-förändringar.

Det gav oss ett rent utgångsläge innan projektet börjar ändras.

---



# 20. Kontrollera projektets filer

Kör:

```bash
ls
```

I legacyprojektet hittades bland annat:

```text
Dockerfile
docker-compose.yml
Makefile
pyproject.toml
poetry.lock
postgres.env

dags/
data/
src/
tests/
cfg/
```

### Varför?

Det bekräftar dels att vi står i rätt katalog, dels vilka tekniker projektet redan använder.

Exempelvis:

```text
Dockerfile
→ hur projektets Docker-image byggs

docker-compose.yml
→ vilka Docker-tjänster som ska köras tillsammans

pyproject.toml
→ Pythonprojektets konfiguration/dependencies

poetry.lock
→ låsta dependency-versioner

dags/
→ Airflow DAG:ar

src/
→ Pythonkod

tests/
→ tester
```

---



# 21. Kontrollera legacyprojektets Pythonkrav

Vi läste Python-relaterade rader utan att ändra filen:

```bash
grep -n "python" pyproject.toml
```

Där fanns bland annat:

```toml
python = "~3.10"
```

### Varför gjorde vi detta?

För att **undersöka projektet innan vi installerar saker**.

Vi konstaterade att legacyprojektet vill använda Python 3.10, medan utvecklingsmiljön har en nyare Pythonversion.

Eftersom projektet ska moderniseras valde vi därför att **inte köra `poetry install` ännu**.

---

# Vad installerades INTE?

Vi installerade medvetet inte:

```text
Apache Airflow
PostgreSQL Server
MinIO
Ollama
Pandas
LangChain
BeautifulSoup
m.fl.
```

direkt i Ubuntu.

Tanken är istället ungefär:

```text
Ubuntu / WSL
│
├── Git
├── Poetry
├── Make
├── VS Code
└── Docker
       │
       ├── Airflow
       ├── PostgreSQL
       ├── MinIO
       └── ev. Ollama
```

Pythonbiblioteken ska dessutom hanteras av projektets dependency-system/container istället för att installeras godtyckligt globalt.

---



# Slutresultatet

Efter dagens arbete har vi alltså en fungerande **grundläggande Data Engineering-utvecklingsmiljö**:

```text
WINDOWS
│
├── Docker Desktop                    ✅
│
└── WSL 2                             ✅
     │
     └── Ubuntu                       ✅
          │
          ├── Docker CLI              ✅
          ├── Docker Compose          ✅
          ├── Git                     ✅
          ├── Make                    ✅
          ├── build-essential         ✅
          ├── curl                    ✅
          ├── Python                  ✅
          ├── Poetry                  ✅
          │
          └── VS Code + WSL           ✅
                 │
                 └── Git-projekt      ✅
```

Och viktigast av allt: Docker testades faktiskt genom att **hämta och köra en container**, inte bara genom att kontrollera versionsnumret.

Vi har däremot **inte gjort projektets modernisering ännu**. Vi har förberett datorn så att den är redo för den.

Nästa tekniska steg bör komma **efter gruppens beslut om framför allt Python- och Airflowversion samt den nya Compose-arkitekturen**. Då kan själva projektarbetet börja utan att vi först behöver reda ut utvecklingsmiljön.
